# Lab 10 — Prefill, Decode, and KV Cache
**Goal:** Understand why autoregressive inference has different memory behavior from training and how KV cache grows with context.

Use the loop: predict → run → inspect → break → decide.


In [ ]:
B,H,T,D=1,32,4096,128
bytes_per=2
kv=2*B*H*T*D*bytes_per
print("KV cache GiB:",kv/1024**3)


## Prediction
Predict how KV memory changes when context length doubles.


In [ ]:
for T in [1024,2048,4096,8192,16384]:
    kv=2*B*H*T*D*bytes_per
    print(T,kv/1024**3)


## Interpretation
KV cache grows approximately linearly with sequence length for a fixed number of KV heads and head dimension.


In [ ]:
# GQA-like illustration: 8 KV heads instead of 32.
for kv_heads in [32,16,8,4]:
    kv=2*B*kv_heads*8192*D*bytes_per
    print("KV heads",kv_heads,"cache GiB",kv/1024**3)


## Answers
Inference systems therefore care about context length, KV head count, precision, batching and memory management. vLLM uses PagedAttention and continuous batching among several techniques for efficient serving.

### Primary source
https://docs.vllm.ai/en/stable/


## Deepening: cache memory grows with workload

This section turns the notebook into a quantitative lab. Predict before executing.


In [ ]:
def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_per_element):
    return 2 * layers * kv_heads * head_dim * bytes_per_element

layers, kv_heads, head_dim, bpe = 32, 8, 128, 2
per_token = kv_bytes_per_token(layers, kv_heads, head_dim, bpe)
print("KV bytes/token:", per_token)
for context in [512, 2048, 8192]:
    for batch in [1, 8]:
        gb = per_token * context * batch / 1e9
        print(f"context={context:5d} batch={batch:2d} approx_KV_GB={gb:.3f}")


## Break experiment

Increase both context and concurrency.

Predict which term dominates:

**weights, KV cache, or runtime overhead**.

Then compare the prediction with a real generation benchmark.

### Report

model + precision + context + concurrency + TTFT + ITL + peak memory + output tok/s

Complete templates/EXPERIMENT_CARD.md for the final result.


## 7. Research exercise — real prompts, real prefill/decode behavior


In [ ]:
!pip -q install -U datasets transformers
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch,time

tok=AutoTokenizer.from_pretrained("sshleifer/tiny-gpt2")
model=AutoModelForCausalLM.from_pretrained("sshleifer/tiny-gpt2")
device="cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

ds=load_dataset("HuggingFaceFW/fineweb","sample-10BT",split="train",streaming=True)
texts=[]
for row in ds:
    t=row.get("text","").strip()
    if len(t)>200:
        texts.append(t[:2000])
    if len(texts)>=8: break

for t in texts[:4]:
    inputs=tok(t,return_tensors="pt",truncation=True,max_length=512).to(device)
    if device=="cuda": torch.cuda.synchronize()
    t0=time.perf_counter()
    with torch.no_grad(): out=model.generate(**inputs,max_new_tokens=32,use_cache=True)
    if device=="cuda": torch.cuda.synchronize()
    print("prompt_tokens=",inputs.input_ids.shape[1],"new_tokens=32","wall_s=",round(time.perf_counter()-t0,3))


<details><summary><strong>Expected findings / answer</strong></summary>

Longer prompts increase prefill work; longer generated continuations increase decode work. A single end-to-end latency number mixes the two phases, so report prompt length, generated length, TTFT/prefill time, and decode throughput separately when possible. The tiny model is only a measurement vehicle; its absolute performance is not representative of a production LLM.
</details>
